# Task 1 — Topological spatial relation qualification

Reproduction of the experiment done by Ji et al (2025) using Jev by TypesafeAI instead of GPT-4. The same data and evaluation methods are used. Prompts are adapted to the Jev model and different prompting strategies (instructions and criteria) are tested.

Input: Two WKT geometries
Output: Topological predicate


Ji, Y., Gao, S., Nie, Y., Majić, I., & Janowicz, K. (2025). Foundation models for geospatial reasoning: Assessing the capabilities of large language models in understanding geometries and topological spatial relations. International Journal of Geographical Information Science, 39(9), 1866–1903. https://doi.org/10.1080/13658816.2025.2511227

## Imports and functions

In [1]:
import pandas as pd
from tqdm import tqdm
from tqdm.asyncio import tqdm 
from datetime import datetime
import os
import json
import numpy as np
from dotenv import load_dotenv
x = load_dotenv()

from typesafe_sdk import Choice, TypeSafeClient, AsyncTypeSafeClient

In [16]:
def format_duration(td):
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    seconds = total_seconds % 60
    return f"{hours:02}:{minutes:02}:{seconds:02} (h:m:s)"

def create_metadata(instructions, criteria, task, criteria_dict, instructions_dict, model):
    meta = {}
    meta['Title'] = f'Task{task}_ins-{instructions}_cri-{criteria}'
    meta['Date'] = datetime.now().strftime('%Y/%m/%d %H:%M')
    meta['Task'] = task

    meta['instructions_id'] = instructions
    meta['instructions'] = instructions_dict[instructions]
    meta['criteria_id'] = criteria
    meta['criteria'] = criteria_dict[criteria]
    meta['model'] = model

    path = f"results/{meta['Title']}.json"
    i = 0

    while os.path.exists(path):
        i += 1
        path = f"results/{meta['Title']}"
        path = f"{path}_{i}.json"

    with open(f'{path}', 'x', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {path}')

    return path

async def async_jev(instructions, criteria, state, model):
    starttime = datetime.now()
    response = await client.system_one(
        model=model,
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time

# Main function. Includes the entire workflow after metadata creation including data saving.
async def async_jev_main(instructions, criteria, criteria_dict, instructions_dict, model, path):
    results = []
    tasks = []
    
    for _, row in relations_test.iterrows():
        if instructions == 'ji-fewshot':
            examples = get_examples(row)
            state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}. Examples: {examples}"
        elif instructions == 'ji-fewshot-CoT':
            examples = get_examples_cot(row)
            state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}. Examples: {examples}"
        else:
            state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"
        
        task = async_jev(instructions_dict[instructions], criteria_dict[criteria], state, model)
        tasks.append(task)

    results_raw = await tqdm.gather(*tasks, desc="Prompting")

    for i, (response, time) in enumerate(results_raw):
        row = relations_test.iloc[i] # Match the result to the original row
        result = {
            'GeomA': row.geometry,
            'GeomB': row.geometry_subject,
            'predicate': row.predicate,
            'choice': response.answers['Task'].choice,
            'probabilities': str(response.answers['Task'].probabilities),
            'confidence': response.answers['Task'].confidence,
            'correct': int(response.answers['Task'].choice == row.predicate),
            'input_tokens': response.usage.input_tokens,
            'output_tokens': response.usage.output_tokens,
            'time': time
        }
        results.append(result)

    results_df = pd.DataFrame(results)
    save_data(path, results_df)

def cost_estimation(instructions, criteria, criteria_dict, instructions_dict):
    inst = instructions_dict[instructions]
    crit = criteria_dict[criteria]
    total = 0
    for i, row in relations_test.iterrows():
        relations_example = relations[relations['idx']>=200].reset_index(drop=True)
        if instructions == 'ji-fewshot':
            examples = get_examples(row)
            state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}. Examples: {examples}"
        elif instructions == 'ji-fewshot-CoT':
            relations_example = relations_example[relations_example['idx']==200]
            examples = get_examples_cot(row)
            state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}. Examples: {examples}"
        else:
            state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"

        prompt = f"{inst} {crit} {state}"
        total += len(prompt) // 3
    return total

def print_summary(metadata):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)
    if os.path.exists(metadata.replace('.json', '.csv')):
        results_df = pd.read_csv(metadata.replace('.json', '.csv'))
    else:
        print('No results available yet.')
        return

    high_conf = results_df[results_df['confidence'].astype(float)>=0.8]
    mid_conf = results_df[(results_df['confidence'].astype(float)<0.8) & (results_df['confidence'].astype(float)>=0.5)]
    low_conf = results_df[results_df['confidence'].astype(float)<0.5]

    summary = f"""
        Summary of results for {meta['Title']}:
        Date: {meta['Date']}
        Tokens used: {int(meta['Input Tokens']) + int(meta['Output Tokens'])}
        Time taken (synchronous): {meta['Duration']}

        Total: {results_df.shape[0]} results, {round(results_df['correct'].mean()*100, 2)}% correct.
        Confidence above 80%: {high_conf.shape[0]} results, {round(high_conf['correct'].mean()*100, 2)}% correct.
        Confidence between 50 and 80%: {mid_conf.shape[0]} results, {round(mid_conf['correct'].mean()*100, 2)}% correct.
        Confidence below 50%: {low_conf.shape[0]} results, {round(low_conf['correct'].mean()*100, 2)}% correct.\n
    """
    return summary

def save_data(metadata, results_df):
    with open(metadata, 'r', encoding='utf-8') as js:
        meta = json.load(js)
    results_df.to_csv(metadata.replace('.json', '.csv'), index=False)

    meta['Input Tokens'] = int(results_df['input_tokens'].astype(int).sum())
    meta['Output Tokens'] = int(results_df['output_tokens'].astype(int).sum())
    meta['Duration'] = format_duration(pd.to_timedelta(results_df['time']).sum())
    meta['Accuracy'] = f'{round(results_df['correct'].mean()*100, 2)}%'
    meta['Results'] = results_df.shape[0]

    with open(f'{metadata}', 'w', encoding='utf-8') as f:
        json.dump(meta, f, indent=4, ensure_ascii=False)
        print(f'Saved {metadata}')

    print(print_summary(metadata))

    return metadata

# Gets examples from relations_example for few-shot prompting. Input is a row from relations_test. Copied from Ji, with minor changes to fit my code
def get_examples(row):
    
    np.random.seed()
    geom_type_subject = row['geom_type_subject']
    geom_type = row['geom_type']
    # create the example set
    df = relations_example[(relations_example['geom_type_subject']==geom_type_subject) &
                            (relations_example['geom_type']==geom_type)]
    examples_df = df[['geometry_subject', 'geometry', 'predicate']].sample(5)
    examples = f""
    for _, row in examples_df.iterrows():
        examples += f"Geometry A: {row['geometry']}; Geometry B: {row['geometry_subject']}; Answer: {row['predicate']}\n"
    return examples

# Gets examples from relations_example for few-shot prompting in CoT config. Copied from Ji, no changes. Used in get_example_cot
def get_cot_example(row):
    relate = list(row['relate'])
    WKT_A = row['geometry_subject']
    WKT_B = row['geometry']
    Type_A = row['geom_type_subject']
    Type_B = row['geom_type']

    example = f"""
First, let's determine the geometry types of A and B.
Geometry A is a {Type_A}.
Geometry B is a {Type_B}.

"""
    predicate = None
    # within = False
    # contains = False

    idx = 1
    IA_IB = 'No' if row['predicate'] in ['disjoint', 'touches'] else 'Yes' #relate[0]=='F' else 'Yes'
    example += f"{idx}. Does the Interior of A intersect the Interior of B? \n{IA_IB}\n"
    idx+=1



    if IA_IB == 'No':
        BA_BB = 'No' if row['predicate']=='disjoint' else 'Yes' #relate[4]=='F' else 'Yes'
        example += f"{idx}. Does A intersect B? \n{BA_BB}\n"
        idx+=1

        if BA_BB=='No':
            predicate = 'disjoint'
        else:
            predicate = 'touches'

        example += f"\nSo {Type_A} A {predicate} {Type_B} B.\n"


    else:
        IA_IB = 'Yes' if (row['predicate'] in ['within', 'equals']) else 'No'
        example += f"{idx}. Does the intersection of A and B equal to  A? \n{IA_IB}\n"
        idx+=1

        if IA_IB=='Yes':
            contains = "No" if row['predicate']=='Within' else "Yes"
            example += f"{idx}. Does the intersection of  A and  B equal to  B? \n{contains}\n"
            idx += 1
            if contains=="No":
                predicate = 'within'
                example += f"So {Type_A} A {predicate} {Type_B} B.\n"

            else:
                predicate = 'equals'
                example += f"So {Type_A} A is {predicate} {Type_B} B.\n"


        if (IA_IB=='No'):
            IA_IB = 'Yes' if (row['predicate'] in ['contains']) else 'No'
            example += f"{idx}. Does the intersection of  A and  B equal to  B? \n{IA_IB}\n"
            idx+=1

            if IA_IB=='Yes':
                predicate = 'contains'
                example += f"So {Type_A} A {predicate} {Type_B} B.\n"
            else:
                Dim = 'No' if (row['predicate']=='crosses') else 'Yes'

                # example += f"""{idx}. Let Dim(A) be the dimension of the Interior of A,
                # Dim(B) be the dimension of the Interior of B, Dim(AB) be the dimension of
                # the Interior of the intersection of A and B. Does Dim(AB)=max(Dim(A), Dim(B))-1 hold? \n{Dim}\n"""
                example += f"""{idx}. Are A an B of the same dimension and
                  their intersection set results in a geometry different from both but of the same dimension?\n{Dim}\n"""
                idx+=1

                if Dim=='No':
                    predicate = 'crosses'
                else:
                    predicate = 'overlaps'
                example += f"So {Type_A} A {predicate} {Type_B} B.\n"


    example += f"ANSWER: ({Type_A}, {predicate}, {Type_B})"

    # print('---------')
    # print(row['predicate'], predicate)
    # print(example)

    return example

# Function get_examples adapted for CoT. Copied from Ji with the same changes as above.
def get_examples_cot(row):
    geom_type_subject = row['geom_type_subject']
    geom_type = row['geom_type']
    # create the example set
    df = relations_example[(relations_example['geom_type_subject']==geom_type_subject) &
                            (relations_example['geom_type']==geom_type)]

    df['example'] = df.apply(lambda row: get_cot_example(row), axis=1)

    examples_df = df[['geometry_subject', 'geometry', 'example']].sample(5)

    examples = f""
    for _, row in examples_df.iterrows():
        examples += f"Geometry A: {row['geometry']}; Geometry B: {row['geometry_subject']}; Reasoning: {row['example']}\n"

    return examples

## Data and Prompts

In [3]:
# Same data as in Ji

relations = pd.read_csv('relations.csv')
relations_test = relations[relations['idx']<40].reset_index(drop=True)
relations_example = relations[relations['idx']>=200].reset_index(drop=True)
# relations_example = relations_example[relations_example['idx']==200]

### Original prompts by Ji
These are the prompts Ji used in their paper taken from the provided code (Code/task1/task1-GPT-4.ipynb).  
Accompanying the prompt-originals are their performance with GPT-4 and information if they have been translated for use in my reproduction. My translations are very close to the original, fixing minor errors and removing information irrelevant for a typesafe model.

Notes:  
CoT is contradictive to system-1-intelligence. I still tested it.  
The 'dimension requirement' was included in all later prompts by Ji. I did so too.  
With few-shot prompts the examples are added to the state during execution.  
CoT few-shot and normal few-shot have differently structured examples.

#### Zero Shot without dimension requirement
Accuracy: 0.632  
Translated to: ji-0shot

SYSTEM_PROMPT = """You will be given the WKT format of geometries given the subject A and reference object B.
Classify their spatial relations into one of the following predicates: contains, within, touches, equals, crosses, disjoint, overlaps.
The predicates are defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.

- Return in the format (Geometry type A, PREDICATE, geometry type B), and nothing else. Geometry types are Point, LineString, Polygon.
- MAKE SURE the PREDICATE in your output is one of the seven predicates stated."""

USER_PROMPT = """
Geometry A: {WKT_A}
Geometry B: {WKT_B}
"""


#### Zero-Shot with dimension requirement
Accuracy: 0.635
Translated to: ji-0shot-dim

SYSTEM_PROMPT = """You will be given the WKT format of geometries given the subject A and reference object B.
Classify their spatial relations into one of the following predicates: contains, within, touches, equals, crosses, disjoint, overlaps.
The predicates are defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.

- Return in the format (Geometry type A, PREDICATE, geometry type B), and nothing else. Geometry types are Point, LineString, Polygon.
- MAKE SURE the PREDICATE in your output is one of the seven predicates stated.
- MAKE SURE the PREDICATE satisfies the the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B.
"""

USER_PROMPT = """
Geometry A: {WKT_A}
Geometry B: {WKT_B}
"""

#### Few-Shot
Accuracy: 0.666 (winner)
Translated to: ji-fewshot

prefix = """
You will be given the WKT format of geometries given the subject A and reference object B.
Classify their spatial relations into one of the following predicates: contains, within, touches, equals, crosses, disjoint, overlaps.
The predicates are defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.

- Return in the format (Geometry type A, PREDICATE, geometry type B), and nothing else. Geometry types are Point, LineString, Polygon.
- MAKE SURE the PREDICATE in your output is one of the seven predicates stated.
- MAKE SURE the PREDICATE satisfies the the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B.
- TRY YOUR BEST to understand how the spatial relations in the examples are determined.


[EXAMPLES]
"""

suffix = """
[END EXAMPLES]

Geometry A: {WKT_A}
Geometry B: {WKT_B}
Desired output:
"""


#### Zero-Shot CoT
Accuracy: 0.610
Translated to: ji-0shot-CoT

SYSTEM_PROMPT = """You will be given the WKT format of geometries given the subject A and reference object B.
Classify their spatial relations into one of the following predicates: contains, within, touches, equals, crosses, disjoint, overlaps.
The predicates are defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.

- MAKE SURE the rationale is complete, professional, consistent, coherent, and accurate to derive the answer for the classification task.
- MAKE SURE the output ends with the 'ANSWER: (Geometry Type A, PREDICATE, Geometry Type B)', where the [Predicate] is one of the seven predicates stated,
and the Geometry types are Point, LineString, Polygon.
- MAKE SURE the PREDICATE satisfies the the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B.
"""

USER_PROMPT = """
Geometry A: {WKT_A}
Geometry B: {WKT_B}
"""

AI_PROMPT = """
Let's think step by step.
"""


#### Few-Shot CoT
Accuracy: 0.627
Translated to: ji-fewshot-CoT

prefix = """You will be given the WKT format of geometries given the subject A and reference object B.
Classify their spatial relations into one of the following predicates: contains, within, touches, equals, crosses, disjoint, overlaps.
The predicates are defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.

- MAKE SURE the rationale is complete, professional, consistent, coherent, and accurate to derive the answer for the classification task.
- MAKE SURE the output ends with the 'ANSWER: (Geometry Type A, PREDICATE, Geometry Type B)', where the PREDICATE is one of the seven predicates stated,
and the Geometry Types are Point, LineString, Polygon.
- MAKE SURE the PREDICATE satisfies the the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B.
- MAKE SURE the same decision process as shown in the examples is used to reason about the answer.
- DO NOT ask repeated questions.

[EXAMPLES]
"""

"""You will be given the WKT format of geometries given the subject A and reference object B.
Classify their spatial relations into one of the following predicates: contains, within, touches, equals, crosses, disjoint, overlaps.
The predicates are defined by DE-9IM and Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.

MAKE SURE the rationale is complete.
MAKE SURE the answer ends with the 'ANSWER: (geometry type A, predicate, geometry type B)', where the Geometry types are Point, LineString, Polygon.
and the predicate is one of the six predicates stated. """

suffix = """
[END EXAMPLES]

Geometry A: {WKT_A}
Geometry B: {WKT_B}
Desired output:
"""

In [4]:
instructions_dict = {
    'basic': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM.",
    'ji-0shot': "You will be given the WKT format of geometries given the subject A and reference object B. Classify their spatial relations into one of the predicates defined by DE-9IM and the Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'.",
    'ji-0shot-dim': "You will be given the WKT format of geometries given the subject A and reference object B. Classify their spatial relations into one of the predicates defined by DE-9IM and the Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'. MAKE SURE the PREDICATE satisfies the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B.",
    'ji-fewshot': "You will be given the WKT format of geometries given the subject A and reference object B and some solved examples to learn from. Classify their spatial relations into one of the predicates defined by DE-9IM and the Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'. MAKE SURE the PREDICATE satisfies the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B.",
    'ji-0shot-CoT': "You will be given the WKT format of geometries given the subject A and reference object B. Classify their spatial relations into one of the predicates defined by DE-9IM and the Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'. Let's think step by step.",
    'ji-fewshot-CoT': "You will be given the WKT format of geometries given the subject A and reference object B and some solved examples to learn from. Classify their spatial relations into one of the predicates defined by DE-9IM and the Open Geospatial Consortium. If A equals to B, there is no need to return 'within' or 'contains'. MAKE SURE the PREDICATE satisfies the dimension requirements defined by DE-9IM and OGC given Geometry Type A and Geometry Type B. Let's think step by step.",
}

In [6]:
criteria_dict = {
    'basic': {
        "contains": "",
        "within": "",
        "touches": "",
        "crosses": "",
        "disjoint": "",
        "overlaps": "",
        "equals": "",
    }
}

In [98]:
# Tests

# Data tests
# print(relations_test.shape[0])
# relations_test[['objectid', 'geom_type', 'geom_type_subject', 'predicate']].groupby(['geom_type_subject', 'predicate', 'geom_type']).count()

# Example of Few-Shot examples
# print(get_examples(relations_test.iloc[0]))

# Example of Few-Shot CoT examples
print(get_examples_cot(relations_test.iloc[0]))

Geometry A: LINESTRING (-89.3576 43.0311, -89.3576 43.0312); Geometry B: LINESTRING (-89.3576 43.0311, -89.3576 43.03117, -89.3576 43.0312); Reasoning: 
First, let's determine the geometry types of A and B.
Geometry A is a LineString.
Geometry B is a LineString.

1. Does the Interior of A intersect the Interior of B? 
Yes
2. Does the intersection of A and B equal to  A? 
Yes
3. Does the intersection of  A and  B equal to  B? 
Yes
So LineString A is equals LineString B.
ANSWER: (LineString, equals, LineString)
Geometry A: LINESTRING (-89.2982 43.095, -89.2982 43.0946); Geometry B: LINESTRING (-89.2965 43.095, -89.2966 43.0935, -89.2982 43.0936, -89.2982 43.095, -89.2965 43.095); Reasoning: 
First, let's determine the geometry types of A and B.
Geometry A is a LineString.
Geometry B is a LineString.

1. Does the Interior of A intersect the Interior of B? 
Yes
2. Does the intersection of A and B equal to  A? 
No
3. Does the intersection of  A and  B equal to  B? 
Yes
So LineString A conta

## Execution

In [ ]:
# Dry test and cost estimation
# Last estimation undershot by 50 cent (4 characters = 1 token). Changed to 3 character per token and it only undershot by 10 cents.
model = 'jev-1.13.0'

estimate = 0
price = 0.042 / 1000000
repeats = 10

for ins in instructions_dict.keys():
    for crit in criteria_dict.keys():
        this_run = cost_estimation(ins, crit, criteria_dict, instructions_dict)
        estimate += this_run
        print(f"Instructions: {ins}, Criteria: {crit}, Tokens: {this_run:,}, Cost: {this_run*price:.2f}$")
            

print(f"""
    Total tokens: {estimate*repeats:,}
    Total cost: {estimate*price*repeats:.2f}$
    """)

Instructions: basic, Criteria: basic, Tokens: 227,157, Cost: 0.01$
Instructions: ji-0shot, Criteria: basic, Tokens: 291,557, Cost: 0.01$
Instructions: ji-0shot-dim, Criteria: basic, Tokens: 352,222, Cost: 0.01$
Instructions: ji-fewshot, Criteria: basic, Tokens: 968,091, Cost: 0.04$
Instructions: ji-0shot-CoT, Criteria: basic, Tokens: 303,695, Cost: 0.01$
Instructions: ji-fewshot-CoT, Criteria: basic, Tokens: 1,744,386, Cost: 0.07$

    Total tokens: 38,871,080
    Total cost: 1.63$
    


In [8]:
# ASYNCHROUNOUS EXECUTION (running this cell costs money)
# Pre run credits: $4.61

for i in range(repeats):
    print(f'Starting run {i}/{repeats}.')
    for ins in instructions_dict.keys():
        for crit in criteria_dict.keys():
            path = create_metadata(ins, crit, '1', criteria_dict, instructions_dict, model)
            client = AsyncTypeSafeClient()
            await async_jev_main(ins, crit, criteria_dict, instructions_dict, model, path)

Starting run 0/10.
Saved results/Task1_ins-basic_cri-basic.json


Prompting: 100%|██████████| 1400/1400 [00:04<00:00, 294.76it/s]


Saved results/Task1_ins-basic_cri-basic.json

        Summary of results for Task1_ins-basic_cri-basic:
        Date: 2026/10/21 21:07
        Tokens used: 868299
        Time taken (synchronous): 01:07:01 (h:m:s)

        Total: 1400 results, 35.79% correct.
        Confidence above 80%: 350 results, 92.0% correct.
        Confidence between 50 and 80%: 380 results, 20.53% correct.
        Confidence below 50%: 670 results, 15.07% correct.

    
Saved results/Task1_ins-ji-0shot_cri-basic.json


Prompting: 100%|██████████| 1400/1400 [00:04<00:00, 298.23it/s]


Saved results/Task1_ins-ji-0shot_cri-basic.json

        Summary of results for Task1_ins-ji-0shot_cri-basic:
        Date: 2026/10/21 21:07
        Tokens used: 921363
        Time taken (synchronous): 01:07:31 (h:m:s)

        Total: 1400 results, 43.86% correct.
        Confidence above 80%: 322 results, 87.27% correct.
        Confidence between 50 and 80%: 499 results, 27.25% correct.
        Confidence below 50%: 579 results, 34.02% correct.

    
Saved results/Task1_ins-ji-0shot-dim_cri-basic.json


...

Saved results/Task1_ins-ji-fewshot_cri-basic_9.json

        Summary of results for Task1_ins-ji-fewshot_cri-basic:
        Date: 2026/10/21 21:13
        Tokens used: 2355162
        Time taken (synchronous): 01:07:20 (h:m:s)

        Total: 1400 results, 36.29% correct.
        Confidence above 80%: 44 results, 70.45% correct.
        Confidence between 50 and 80%: 409 results, 48.41% correct.
        Confidence below 50%: 947 results, 29.46% correct.

    
Saved results/Task1_ins-ji-0shot-CoT_cri-basic_9.json


Prompting: 100%|██████████| 1400/1400 [00:04<00:00, 289.60it/s]


Saved results/Task1_ins-ji-0shot-CoT_cri-basic_9.json

        Summary of results for Task1_ins-ji-0shot-CoT_cri-basic:
        Date: 2026/10/21 21:13
        Tokens used: 931176
        Time taken (synchronous): 01:08:56 (h:m:s)

        Total: 1400 results, 44.57% correct.
        Confidence above 80%: 317 results, 92.11% correct.
        Confidence between 50 and 80%: 523 results, 27.72% correct.
        Confidence below 50%: 560 results, 33.39% correct.

    
Saved results/Task1_ins-ji-fewshot-CoT_cri-basic_9.json


Prompting: 100%|██████████| 1400/1400 [00:05<00:00, 263.99it/s]

Saved results/Task1_ins-ji-fewshot-CoT_cri-basic_9.json

        Summary of results for Task1_ins-ji-fewshot-CoT_cri-basic:
        Date: 2026/10/21 21:13
        Tokens used: 3014970
        Time taken (synchronous): 01:09:32 (h:m:s)

        Total: 1400 results, 32.43% correct.
        Confidence above 80%: 44 results, 72.73% correct.
        Confidence between 50 and 80%: 431 results, 39.91% correct.
        Confidence below 50%: 925 results, 27.03% correct.

    


## Result Evaluation

I am disregarding geometry type in the experiment in the evaluation. The paper did so also, but it might be interesting (not sure).

In [45]:
# Combine all results to one dataframe

results_list = os.listdir('results')
all_results = []

for result in results_list:
    if result.endswith('.csv'):
        continue
    result = result.rstrip('.json')

    with open(f'results/{result}.json', 'r', encoding='utf-8') as js:
        meta = json.load(js)

    
    results_df = pd.read_csv(f'results/{result}.csv')

    high_conf = results_df[results_df['confidence'].astype(float)>=0.8]
    mid_conf = results_df[(results_df['confidence'].astype(float)<0.8) & (results_df['confidence'].astype(float)>=0.5)]
    low_conf = results_df[results_df['confidence'].astype(float)<0.5]

    result_dict = {
        'Title': meta['Title'],
        'Date': meta['Date'],
        'Instructions': meta['instructions_id'],
        'Criteria': meta['criteria_id'],
        'Input tokens': meta['Input Tokens'],
        'Results': results_df.shape[0],
        'Total %': round(results_df['correct'].mean()*100, 2),
        'High Conf': high_conf.shape[0],
        'High Conf %': round(high_conf['correct'].mean()*100, 2),
        'Mid Conf': mid_conf.shape[0],
        'Mid Conf %': round(mid_conf['correct'].mean()*100, 2),
        'Low Conf': low_conf.shape[0],
        'Low Conf %': round(low_conf['correct'].mean()*100, 2),
    }
    predicates = ["contains","within","touches","crosses","disjoint","overlaps","equals"]
    for pred in predicates:
        df = results_df[results_df['choice'] == pred]
        correct = round(df['correct'].mean()*100, 2)
        result_dict[f'{pred}%'] = float(correct)

    all_results.append(result_dict)

all_results_df = pd.DataFrame(all_results)
all_results_df.sort_values('Instructions')

,Title,Date,Instructions,Criteria,Input tokens,Results,Total %,High Conf,High Conf %,Mid Conf,Mid Conf %,Low Conf,Low Conf %,contains%,within%,touches%,crosses%,disjoint%,overlaps%,equals%
29,Task1_ins-basic_cri-basic,2026/10/21 21:08,basic,basic,764950,1400,36.29,355,90.99,381,20.73,664,15.96,1.72,0.64,48.34,33.33,83.16,51.79,91.25
2,Task1_ins-basic_cri-basic,2026/10/21 21:09,basic,basic,764950,1400,35.57,346,92.20,382,20.68,672,14.88,1.23,0.32,48.03,50.00,82.90,41.07,91.25
27,Task1_ins-basic_cri-basic,2026/10/21 21:07,basic,basic,764950,1400,35.79,350,92.00,380,20.53,670,15.07,1.74,0.31,45.96,33.33,83.99,45.28,93.59
50,Task1_ins-basic_cri-basic,2026/10/21 21:11,basic,basic,764950,1400,35.64,351,91.45,373,21.18,676,14.64,1.73,0.31,47.65,33.33,83.33,45.61,91.03
49,Task1_ins-basic_cri-basic,2026/10/21 21:11,basic,basic,764950,1400,35.64,357,90.48,388,20.36,655,14.81,1.47,0.32,48.34,37.50,83.55,46.55,89.61
47,Task1_ins-basic_cri-basic,2026/10/21 21:13,basic,basic,764950,1400,36.00,354,91.24,386,20.21,660,15.61,1.98,0.32,48.37,33.33,82.95,46.67,90.79
44,Task1_ins-basic_cri-basic,2026/10/21 21:10,basic,basic,764950,1400,35.64,355,90.99,394,20.56,651,14.59,1.00,0.31,49.04,36.36,83.38,41.18,91.03
42,Task1_ins-basic_cri-basic,2026/10/21 21:08,basic,basic,764950,1400,36.07,347,91.64,381,21.78,672,15.48,1.95,0.33,47.97,42.86,83.46,46.55,90.79
33,Task1_ins-basic_cri-basic,2026/10/21 21:10,basic,basic,764950,1400,35.57,351,90.60,388,22.42,661,14.07,1.00,0.31,48.37,44.44,83.07,44.83,92.11
31,Task1_ins-basic_cri-basic,2026/10/21 21:12,basic,basic,764950,1400,35.43,361,90.86,383,19.06,656,14.48,0.99,0.32,47.71,35.71,82.86,46.30,90.79


In [49]:
# Average results for runs with the same conditions

mean_list = []
result_fields = ['Total %', 'High Conf %', 'Mid Conf %', 'Low Conf %', 'contains%', 'within%', 'touches%', 'crosses%', 'disjoint%', 'overlaps%', 'equals%']

for title in all_results_df['Title'].unique():
    df = all_results_df[all_results_df['Title'] == title]
    mean_df = df.drop(columns=all_results_df.keys()[6:]).head(1)
    mean_df['Runs'] = df.shape[0]
    
    for field in result_fields:
        mean = df[field].mean()
        mean_df[field] = round(mean, 2)
    mean_list.append(mean_df)

all_mean = pd.concat(mean_list)
all_mean.sort_values('Total %', ascending=False)

,Title,Date,Instructions,Criteria,Input tokens,Results,Runs,Total %,High Conf %,Mid Conf %,Low Conf %,contains%,within%,touches%,crosses%,disjoint%,overlaps%,equals%
5,Task1_ins-ji-0shot-CoT_cri-basic,2026/10/21 21:08,ji-0shot-CoT,basic,827950,1400,10,44.51,91.39,26.97,33.87,0.00,20.35,66.58,31.15,87.81,37.26,89.71
0,Task1_ins-ji-0shot-dim_cri-basic,2026/10/21 21:09,ji-0shot-dim,basic,860150,1400,10,43.75,72.85,29.44,37.83,0.00,20.87,71.20,37.26,86.79,34.79,88.90
3,Task1_ins-ji-0shot_cri-basic,2026/10/21 21:10,ji-0shot,basic,818150,1400,10,43.70,88.05,27.08,33.84,0.00,19.89,68.19,34.79,85.87,34.62,89.77
6,Task1_ins-ji-fewshot_cri-basic,2026/10/21 21:09,ji-fewshot,basic,2240107,1400,10,36.83,80.15,48.84,29.82,28.53,31.30,39.64,25.99,53.20,19.66,53.89
2,Task1_ins-basic_cri-basic,2026/10/21 21:09,basic,basic,764950,1400,10,35.76,91.24,20.75,14.96,1.48,0.35,47.98,38.02,83.26,45.58,91.22
1,Task1_ins-ji-fewshot-CoT_cri-basic,2026/10/21 21:13,ji-fewshot-CoT,basic,2911152,1400,10,31.78,66.95,40.00,26.70,26.51,60.10,38.62,21.63,41.40,14.82,24.26


In [ ]:
# Evaluation of individual predicate accuracy

results_df = pd.read_csv(f'results/Task1_ins-ji_cri-basic.csv')


pred_results = {}
predicates = ["contains","within","touches","crosses","disjoint","overlaps","equals"]
for pred in predicates:
    df = results_df[results_df['choice'] == pred]
    correct = round(df['correct'].mean()*100, 2)
    pred_results[pred] = float(correct)

pred_results

{'contains': 0.0,
 'within': 19.67,
 'touches': 66.67,
 'crosses': 0.0,
 'disjoint': 84.62,
 'overlaps': 32.89,
 'equals': 89.06}

## Archive

In [ ]:
# Retired Instructions and Criteria

instructions_dict = {
    'think': "What topological predicate best describes the relationship between Place A and Place B? The predicates follow the definition of DE-9IM. Think step by step.", # Thinking step by step is contradictory to system one intelligence. It also did not influence results.
}

criteria_dict = {

}

In [ ]:
# SYNCHRONOUS EXECUTION (waaay slower; use asynchronous)
def ask_jev(instructions, criteria, state):
    starttime = datetime.now()
    response = client.system_one(
        state=state,
        questions={
            "Task": Choice(
                instructions=instructions,
                criteria=criteria,
            )
        },
    )
    endtime = datetime.now()
    time = str(endtime - starttime)

    return response, time

client = TypeSafeClient()

instructions = 'think'
criteria = 'basic'
path = create_metadata(instructions, criteria, '1', criteria_dict, instructions_dict)
results = []

df = pd.read_csv('relations.csv')
df = df[:1]

for row in tqdm(df.itertuples(), total=len(df), desc="Prompting"):
    result = {}
    state = f"Geometry A: {row.geometry}, Geometry B: {row.geometry_subject}"
    response, time = ask_jev(instructions_dict[instructions], criteria_dict[criteria], state)

    result['GeomA'] = row.geometry
    result['GeomB'] = row.geometry_subject 
    result['predicate'] = row.predicate
    result['choice'] = response.answers['Task'].choice
    result['probabilities'] = str(response.answers['Task'].probabilities)
    result['confidence'] = response.answers['Task'].confidence
    result['correct'] = int(result['choice'] == result['predicate'])
    result['input_tokens'] = response.usage.input_tokens
    result['output_tokens'] = response.usage.output_tokens
    result['time'] = time
    
    results.append(result)

results_df = pd.DataFrame(results)
save_data(path, results_df)

Saved results/Task1_ins-think_cri-basic_3.json


Processing geometries: 100%|██████████| 1/1 [00:20<00:00, 20.61s/it]

Saved results/Task1_ins-think_cri-basic_3.json

        Summary of results for Task1_ins-think_cri-basic:
        Date: 2026/10/19 17:34
        Tokens used: 506
        Time taken: 00:00:20 (h:m:s)

        Total: 1 results, 0.0% correct.
        Confidence above 80%: 1 results, 0.0% correct.
        Confidence between 50 and 80%: 0 results, nan% correct.
        Confidence below 50%: 0 results, nan% correct.

    


'results/Task1_ins-think_cri-basic_3.json'

In [ ]:
# Model dump for function create_metadata
async def model_list() -> None:
    async with AsyncTypeSafeClient() as client:
        return await client.models.list()

models = await model_list()
model_dump = models.model_dump()